# Vector Database Tutorial

This comprehensive tutorial demonstrates how to use Milvus, a high-performance vector database, for semantic search and hybrid retrieval applications.

## What You'll Learn

- Setting up and connecting to Milvus
- Creating collections with custom schemas
- Inserting and managing data
- Performing semantic, lexical, and hybrid searches
- Querying with filters
- Best practices and cleanup

## Prerequisites

- Python 3.8+
- Basic understanding of vector databases and embeddings

**Note**: This tutorial uses fully local components - no cloud services required!
- **Milvus Lite**: Local file-based vector database
- **sentence-transformers**: Local embedding model (all-MiniLM-L6-v2)
- **BM25**: Local sparse embeddings for lexical search


## Necessary Theory

- [Embeddings](https://youtu.be/wgfSDrqYMJ4?si=RA4UtxT5xvPmxuXQ)
- [Vector Search Strategies](https://youtu.be/r0Dciuq0knU?si=iz2tmNQD4cCdJ0dK)


## 📋 Prerequisites & Installation

**⚠️ Important**: Before running this notebook, please follow the installation instructions in the [README.md](README.md) file to:

- ✅ Set up your virtual environment
- ✅ Install all required dependencies
- ✅ Configure the `.env` file (optional - defaults provided)

This will ensure all necessary packages are installed and properly configured.

## 1. Import Libraries and Load Configuration

Import necessary libraries and load environment variables:

In [1]:
from utils.credentials_utils import LocalConfig
from utils.milvus_utils import MilvusClient

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Environment Configuration

Create a `.env` file in your project root with the following variables:

```bash
# Local Configuration (no cloud services required!)
MILVUS_DB_PATH=./milvus_demo.db
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
```

**Note**: These are the default values. The tutorial will work even without a `.env` file!

## 3. Initialize Local Milvus Client

Initialize the Milvus Lite client with local configuration:

In [6]:
# Load configuration
config = LocalConfig()

# Initialize Milvus Lite client (local file-based database)
client = MilvusClient(
    db_path=config.milvus_db_path,
    model_name=config.embedding_model
)

print("✓ Connected to Milvus Lite successfully")
print(f"Database path: {config.milvus_db_path}")
print(f"Embedding model: {config.embedding_model}")
print(f"Available collections: {client.list_collections()}")

✓ Connected to Milvus Lite successfully
Database path: ./milvus_demo.db
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Available collections: ['demo_documents']


## 4. Creating a New Collection with its own Schema

Milvus collections are similar to tables in relational databases. Each collection has a schema that defines:
- **Fields**: Data columns (text, vectors, metadata)
- **Vector fields**: Dense and/or sparse embeddings
- **Metadata fields**: Additional attributes for filtering

Let's create a simple collection for storing documents with embeddings:

In [3]:
from pymilvus import CollectionSchema, FieldSchema, DataType

# Define collection name
DEMO_COLLECTION = "demo_documents"

# Define schema fields
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    FieldSchema(name="vector_dense", dtype=DataType.FLOAT_VECTOR, dim=384),  # sentence-transformers/all-MiniLM-L6-v2 produces 384-dim embeddings
    FieldSchema(name="vector_sparse", dtype=DataType.SPARSE_FLOAT_VECTOR),  # For BM25 lexical search
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="timestamp", dtype=DataType.INT64)
]

# Create schema
schema = CollectionSchema(
    fields=fields,
    description="Demo collection for tutorial"
)

# Create collection
client.create_collection(
    collection_name=DEMO_COLLECTION,
    schema=schema
)

print(f"✓ Collection '{DEMO_COLLECTION}' created successfully")
print(f"Available collections: {client.list_collections()}")

✓ Collection 'demo_documents' created successfully
Available collections: ['demo_documents']


## 5. Building Indexes

Indexes improve search performance. For vector fields, Milvus Lite supports:
- **FLAT**: Brute-force exact search (most stable for Milvus Lite, recommended)
- **HNSW**: Hierarchical Navigable Small World, good for approximate nearest neighbor search
- **AUTOINDEX**: Automatically selects the best index (may cause issues with some Milvus Lite versions)

**Important Notes:**
- Milvus Lite does not support IVF_FLAT, IVF_SQ8, SPARSE_INVERTED_INDEX, or other advanced indexes
- Sparse vectors automatically use FLAT index without requiring explicit index creation
- **FLAT is recommended for Milvus Lite** as it's the most stable and works perfectly for small-to-medium datasets (< 1M vectors)
- FLAT provides exact search results with no configuration needed

In [5]:
# Build index for dense vector field
client.build_index(
    collection_name=DEMO_COLLECTION,
    field_name="vector_dense",
    index_type="FLAT",  
    metric_type="COSINE"
)
print("✓ Dense vector index built successfully")

# Milvus Lite automatically uses FLAT index for sparse vectors, which works fine for tutorials
# Sparse vector search will still work perfectly without explicit index creation
print("✓ Sparse vectors will use FLAT index (Milvus Lite default)")

✓ Dense vector index built successfully
✓ Sparse vectors will use FLAT index (Milvus Lite default)


## 6. Inserting Data

Insert documents with their embeddings into the collection:

In [7]:
import json
import time

# Load sample dataset
with open('sample_data.json', 'r') as f:
    data = json.load(f)
    sample_docs = data['documents']

print(f"Loaded {len(sample_docs)} sample documents")
print("\nFirst document example:")
print(f"  Text: {sample_docs[0]['text'][:80]}...")
print(f"  Category: {sample_docs[0]['category']}")
print(f"  Topic: {sample_docs[0]['topic']}")
print(f"  Source: {sample_docs[0]['source']}")

# Extract texts for embedding generation
sample_texts = [doc['text'] for doc in sample_docs]

# Generate dense embeddings for all texts
print(f"\nGenerating dense embeddings for {len(sample_texts)} documents...")
embeddings = client.embeddings.get_dense_embeddings(sample_texts)
print(f"✓ Generated {len(embeddings)} dense embeddings")

# Fit BM25 model and generate sparse embeddings
print("\nFitting BM25 model on corpus...")
client.sparse_embeddings.fit_corpus(sample_texts)
print("✓ BM25 model fitted")

print(f"\nGenerating sparse embeddings for {len(sample_texts)} documents...")
sparse_embeddings = client.sparse_embeddings.get_sparse_embeddings(sample_texts)
print(f"✓ Generated {len(sample_texts)} sparse embeddings")

# Prepare documents for insertion with both dense and sparse vectors
documents = [
    {
        "text": doc['text'],
        "vector_dense": dense_emb,
        "vector_sparse": sparse_emb,
        "category": doc['category'],
        "source": doc['source'],
        "timestamp": int(time.time())
    }
    for doc, dense_emb, sparse_emb in zip(sample_docs, embeddings, sparse_embeddings)
]

# Insert documents in batches for better performance
batch_size = 10
total_inserted = 0

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    result = client.insert_data(
        collection_name=DEMO_COLLECTION,
        documents=batch
    )
    total_inserted += len(batch)
    print(f"  Inserted batch {i//batch_size + 1}: {len(batch)} documents")

print(f"\n✓ Total inserted: {total_inserted} documents")

# Flush to ensure data is persisted
client.flush_collection(DEMO_COLLECTION)
print("✓ Data flushed to storage")

Loaded 20 sample documents

First document example:
  Text: Milvus is an open-source vector database built for scalable similarity search an...
  Category: database
  Topic: vector database
  Source: documentation

Generating dense embeddings for 20 documents...
✓ Generated 20 dense embeddings

Fitting BM25 model on corpus...
✓ BM25 model fitted

Generating sparse embeddings for 20 documents...
✓ Generated 20 sparse embeddings
  Inserted batch 1: 10 documents
  Inserted batch 2: 10 documents

✓ Total inserted: 20 documents
✓ Data flushed to storage


## 9. Semantic Search

Semantic search finds documents based on meaning similarity using vector embeddings:

In [8]:
# Define search query
query = "How do I build a recommendation system?"

# Perform semantic search
results = client.semantic_search(
    collection_name=DEMO_COLLECTION,
    query=query,
    anns_field="vector_dense",
    output_fields=["text", "category"],
    limit=20
)

print(f"\nQuery: '{query}'\n")
print("Top 5 Results:\n" + "="*80)

for i, result in enumerate(results[0], 1):
    print(f"\n{i}. [Score: {result['distance']:.4f}]")
    print(f"   Text: {result['entity']['text'][:100]}...")
    print(f"   Category: {result['entity']['category']}")


Query: 'How do I build a recommendation system?'

Top 5 Results:

1. [Score: 0.5766]
   Text: Recommendation systems use vector similarity to suggest products, content, or services based on user...
   Category: ai applications

2. [Score: 0.2844]
   Text: Fine-tuning embedding models on domain-specific data improves retrieval accuracy for specialized app...
   Category: machine learning

3. [Score: 0.2650]
   Text: Metadata filtering enables precise queries by combining vector similarity with structured data filte...
   Category: database

4. [Score: 0.2646]
   Text: Question answering systems leverage vector databases to retrieve relevant context from knowledge bas...
   Category: ai applications

5. [Score: 0.2473]
   Text: RAG (Retrieval-Augmented Generation) combines vector databases with large language models to provide...
   Category: ai applications

6. [Score: 0.2409]
   Text: Vector embeddings are numerical representations of data that capture semantic meaning. They enable m.

## 10. Lexical Search (BM25)

Lexical search uses keyword matching with BM25 algorithm:

In [9]:
# Lexical Search using BM25 sparse vectors
query = "vector database indexing"

print(f"Query: '{query}'\n")
print("Lexical Search Results (BM25):\n" + "="*80)

# Perform lexical search
results = client.lexical_search(
    collection_name=DEMO_COLLECTION,
    query=query,
    output_fields=["text", "category"],
    limit=5
)

for i, result in enumerate(results[0], 1):
    print(f"\n{i}. [Score: {result['distance']:.4f}]")
    print(f"   Text: {result['entity']['text'][:200]}...")
    print(f"   Category: {result['entity']['category']}")

print("\n" + "="*80)
print("Note: Lexical search finds documents based on exact keyword matches using BM25.")

Query: 'vector database indexing'

Lexical Search Results (BM25):

1. [Score: 2.4980]
   Text: Index types in Milvus include IVF_FLAT for balanced performance, HNSW for speed, and IVF_SQ8 for memory efficiency. Choose based on your use case....
   Category: database

2. [Score: 1.8059]
   Text: Milvus is an open-source vector database built for scalable similarity search and AI applications. It supports billions of vectors with millisecond search latency....
   Category: database

3. [Score: 1.7117]
   Text: Question answering systems leverage vector databases to retrieve relevant context from knowledge bases before generating answers....
   Category: ai applications

4. [Score: 1.7117]
   Text: Real-time search applications require low-latency vector databases that can handle concurrent queries while maintaining high throughput....
   Category: database

5. [Score: 1.6094]
   Text: RAG (Retrieval-Augmented Generation) combines vector databases with large language models to provide acc

## 11. Hybrid Search

Hybrid search combines semantic and lexical search for better results:

In [10]:
# Hybrid Search combining semantic and lexical approaches
query = "machine learning embeddings"

print(f"Query: '{query}'\n")
print("Hybrid Search Results:\n" + "="*80)

# Perform hybrid search
results = client.hybrid_search(
    collection_name=DEMO_COLLECTION,
    query=query,
    dense_field="vector_dense",
    sparse_field="vector_sparse",
    output_fields=["text", "category"],
    dense_weight=0.7,
    sparse_weight=0.3,
    ranker_type="weighted",
    limit=5
)

for i, result in enumerate(results, 1):
    print(f"\n{i}. [Score: {result['distance']:.4f}]")
    print(f"   Text: {result['entity']['text'][:100]}...")
    print(f"   Category: {result['entity']['category']}")

print("\n" + "="*80)
print("Configuration:")
print("  - Dense weight: 0.7 (semantic similarity)")
print("  - Sparse weight: 0.3 (keyword matching)")
print("  - Ranker: Weighted")
print("\nHybrid search combines the best of both semantic and keyword-based search!")

Query: 'machine learning embeddings'

Hybrid Search Results:

1. [Score: 1.9307]
   Text: Machine learning models like BERT, GPT, and sentence transformers can generate high-quality embeddin...
   Category: machine learning

2. [Score: 1.2281]
   Text: Vector embeddings are numerical representations of data that capture semantic meaning. They enable m...
   Category: machine learning

3. [Score: 0.5406]
   Text: Fine-tuning embedding models on domain-specific data improves retrieval accuracy for specialized app...
   Category: machine learning

4. [Score: 0.4836]
   Text: Anomaly detection systems use vector embeddings to identify unusual patterns or outliers in data by ...
   Category: ai applications

5. [Score: 0.2833]
   Text: Milvus is an open-source vector database built for scalable similarity search and AI applications. I...
   Category: database

Configuration:
  - Dense weight: 0.7 (semantic similarity)
  - Sparse weight: 0.3 (keyword matching)
  - Ranker: Weighted

Hybrid se

## 12. Querying with Filters

Use metadata filters to narrow down search results:

In [11]:
# Query with filter expression
results = client.query(
    collection_name=DEMO_COLLECTION,
    filter='category == "ai applications"',
    output_fields=["text", "category", "timestamp"],
    limit=10
)

print(f"Found {len(results)} documents with category='ai applications'\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc['text'][:80]}...")
    print(f"   Category: {doc['category']}")
    print()

Found 6 documents with category='ai applications'

1. RAG (Retrieval-Augmented Generation) combines vector databases with large langua...
   Category: ai applications

2. Recommendation systems use vector similarity to suggest products, content, or se...
   Category: ai applications

3. Image search applications convert images to vectors using CNN models, enabling v...
   Category: ai applications

4. Question answering systems leverage vector databases to retrieve relevant contex...
   Category: ai applications

5. Anomaly detection systems use vector embeddings to identify unusual patterns or ...
   Category: ai applications

6. Duplicate detection uses vector similarity to identify near-duplicate content in...
   Category: ai applications



## 13. Advanced Filter Expressions

Milvus supports complex filter expressions:

In [12]:
# Examples of filter expressions
filter_examples = [
    # Equality
    'category == "tutorial"',
    
    # Inequality
    'timestamp > 1609459200',
    
    # IN operator
    'category in ["tutorial", "example"]',
    
    # Logical operators
    'category == "tutorial" and timestamp > 1609459200',
    'category == "tutorial" or category == "demo"',
    
    # String operations
    'text like "vector%"',  # Starts with "vector"
]

print("Filter Expression Examples:\n")
for i, expr in enumerate(filter_examples, 1):
    print(f"{i}. {expr}")

Filter Expression Examples:

1. category == "tutorial"
2. timestamp > 1609459200
3. category in ["tutorial", "example"]
4. category == "tutorial" and timestamp > 1609459200
5. category == "tutorial" or category == "demo"
6. text like "vector%"


## 14. Semantic Search with Filters

Combine semantic search with metadata filtering:

In [13]:
query = "database technology"

# Semantic search with filter
results = client.semantic_search(
    collection_name=DEMO_COLLECTION,
    query=query,
    anns_field="vector_dense",
    output_fields=["text", "source"],
    limit=5,
    filter='source == "tutorial"'
)

print(f"Query: '{query}' with filter: category='tutorial'\n")
print("Results:\n" + "="*60)

for i, result in enumerate(results[0], 1):
    print(f"\n{i}. [Score: {result['distance']:.4f}]")
    print(f"   {result['entity']['text']}")
    print(f"   Source: {result['entity']['source']}")

Query: 'database technology' with filter: category='tutorial'

Results:

1. [Score: 0.3390]
   Metadata filtering enables precise queries by combining vector similarity with structured data filters like date ranges, categories, or tags.
   Source: tutorial

2. [Score: 0.2223]
   Fine-tuning embedding models on domain-specific data improves retrieval accuracy for specialized applications like medical or legal search.
   Source: tutorial

3. [Score: 0.1631]
   Vector embeddings are numerical representations of data that capture semantic meaning. They enable machines to understand and compare content based on similarity.
   Source: tutorial

4. [Score: 0.0984]
   Machine learning models like BERT, GPT, and sentence transformers can generate high-quality embeddings from text, images, and other data types.
   Source: tutorial

5. [Score: 0.0451]
   Cosine similarity measures the angle between vectors, making it ideal for comparing embeddings. Values range from -1 to 1, with 1 indicating ide

## 15. Best Practices

### Performance Optimization

1. **Batch Operations**: Insert/update data in batches (1000-5000 records)
2. **Index Selection**: Choose appropriate index type based on your use case
3. **Field Selection**: Only retrieve fields you need with `output_fields`
4. **Connection Pooling**: Reuse client connections

### Data Management

1. **Schema Design**: Plan your schema carefully - it's hard to change later
2. **Metadata**: Use metadata fields for filtering and organization
3. **Flush Regularly**: Call `flush_collection()` after bulk inserts
4. **Monitor Size**: Keep track of collection size and performance

### Search Optimization

1. **Limit Results**: Use appropriate `limit` values
2. **Filter First**: Apply filters before vector search when possible
3. **Hybrid Search**: Combine semantic and lexical for best results
4. **Tune Weights**: Experiment with dense/sparse weights in hybrid search

## 16. Cleanup

Remove demo collection when done:

In [14]:
# Delete the demo collection
# Uncomment to actually delete
client.delete_collection(DEMO_COLLECTION)
print(f"✓ Collection '{DEMO_COLLECTION}' deleted")

✓ Collection 'demo_documents' deleted


## 17. Summary and Next Steps

### What We Covered

✓ Setting up Milvus connection  
✓ Creating collections with schemas  
✓ Building indexes for performance  
✓ Inserting data  
✓ Semantic search with embeddings  
✓ Filtering and querying  
✓ Best practices and optimization  

### Resources

- [Milvus Documentation](https://milvus.io/docs)
- [PyMilvus API Reference](https://milvus.io/api-reference/pymilvus/v2.3.x/About.md)
- [Milvus GitHub](https://github.com/milvus-io/milvus)
- [Community Forum](https://discuss.milvus.io/)
- [IBM YouTube Channel](https://youtu.be/r0Dciuq0knU?si=iz2tmNQD4cCdJ0dK)

## Appendix: Utility Functions Reference

### MilvusClient Methods

```python
# Collection Management
client.create_collection(collection_name, schema)
client.delete_collection(collection_name)
client.list_collections()
client.flush_collection(collection_name)

# Index Management
client.build_index(collection_name, field_name, index_type, metric_type)

# Data Operations
client.insert_data(collection_name, documents)
client.upsert(collection_name, documents)
client.query(collection_name, filter, limit, output_fields)

# Search Operations
client.semantic_search(collection_name, query, anns_field, output_fields, limit)
client.lexical_search(collection_name, query, anns_field, output_fields, limit)
client.hybrid_search(collection_name, query, dense_field, sparse_field, ...)

# Embedding Operations
client.embeddings.get_dense_embeddings(texts)
client.get_dense_vector(query)
```

### Common Metric Types

- **COSINE**: Cosine similarity (recommended for most cases)
- **L2**: Euclidean distance
- **IP**: Inner product
- **BM25**: For lexical/sparse vectors

### Common Index Types (Milvus Lite Compatible)

- **FLAT**: Exact brute-force search (recommended for Milvus Lite, most stable)
  - Works perfectly for small-to-medium datasets (< 1M vectors)
  - No configuration needed
  - Provides exact search results
  - Automatically used for sparse vectors
- **HNSW**: Hierarchical Navigable Small World, fast approximate search, higher memory
  - Good for larger datasets where approximate results are acceptable
- **AUTOINDEX**: Automatically selects best index
  - May cause internal errors with some Milvus Lite versions
  - Use FLAT instead for maximum stability

**Note**: Full Milvus server supports additional types like IVF_FLAT, IVF_SQ8, and SPARSE_INVERTED_INDEX, but Milvus Lite is limited to the above. For Milvus Lite, **FLAT is the recommended choice** as it's the most stable and reliable option.